# The Sample Proportion

How many of Statistopia's ten inhabitants own a bicycle? Four — so the **population proportion** is $p = \frac{4}{10} = 40\%$. Without access to the full population, you estimate it from a random sample: two bike owners among six sampled people give the **sample proportion** $\hat{p} = \frac{2}{6} = 33.3\%$. This notebook formalizes the pair $(p, \hat{p})$ — and shows the neat unification: a proportion is just the *mean of a yes/no variable*, so everything from the sample-mean notebook applies verbatim. It is based on the third video of Lesson 5 (Sampling and Point Estimation).

## 1. Introduction

Means aren't the only population quantity worth estimating. *What fraction of users click? What share of parts are defective? What percentage of voters favor a candidate?* — these are **proportions**, and they are estimated from samples exactly like means, with their own notation.

After completing this notebook you will be able to:

- Compute the **population proportion** $p$ and the **sample proportion** $\hat{p}$.
- Recognize a proportion as the mean of a 0/1 (indicator) variable, inheriting all sample-mean properties.
- Simulate the sampling distribution of $\hat{p}$ and its $\sqrt{p(1-p)/n}$ spread.
- Connect $\hat{p}$ to the binomial distribution from Lesson 2.

## 2. Theory

### The population proportion $p$

Statistopia again, $N = 10$, and everybody owns exactly one form of transportation: a car or a bicycle. Four people own bicycles, so the fraction of bike owners is

$$
p = \frac{\text{number of items with the characteristic}}{N} = \frac{4}{10} = 0.4
$$

This is the **population proportion**, denoted $p$ — for any characteristic $x$ (owning a bike, clicking an ad, being defective), $p = \frac{\#\{x\}}{N}$.

### The sample proportion $\hat{p}$

Without access to all 10 people, take a random sample of $n = 6$. If 2 of them own bikes,

$$
\hat{p} = \frac{2}{6} = 0.333
$$

This is the **sample proportion**, denoted $\hat{p}$ ("p-hat") — an *estimate* of the population proportion $p$. Like the sample mean, it varies from sample to sample and improves with $n$.

### A proportion is a mean in disguise

Code each person as $1$ (bike) or $0$ (car). Then:

- the population proportion is the population **mean** of these 0/1 values ($\frac{0+1+\cdots}{N} = \frac{4}{10}$);
- the sample proportion is the sample mean of the sampled 0/1 values.

So every fact from the sample-mean notebook transfers instantly: $\hat{p}$ is **unbiased** (its average over all random samples is $p$), needs **random i.i.d. sampling**, and its spread shrinks like $1/\sqrt{n}$ — specifically $\sqrt{p(1-p)/n}$, since a 0/1 variable has variance $p(1-p)$ (the Bernoulli variance from Lesson 3).

### The binomial connection

The *count* of bike owners in an i.i.d. sample of size $n$ is a Binomial$(n, p)$ variable (Lesson 2!). The sample proportion is that count divided by $n$ — which is why its distribution is a rescaled binomial.

## 3. Worked Examples

### Example 1: the video's numbers

$p = \frac{4}{10} = 40\%$ of Statistopians own bicycles; the sample of 6 with 2 bike owners estimates $\hat{p} = 33.3\%$. The estimate is off by 6.7 points — reasonable for $n = 6$.

### Example 2: possible values of $\hat{p}$

With $n = 6$, the sample proportion can only be $0, \tfrac16, \tfrac26, \dots, 1$ — a discrete ladder. Small samples can't even *express* 40%: the nearest rungs are $\tfrac26 = 33.3\%$ and $\tfrac36 = 50\%$. Larger $n$ makes the ladder finer as well as the estimate tighter.

### Example 3: an election poll

A candidate's true support is $p = 0.52$. A poll of $n = 1{,}000$ voters has standard error $\sqrt{0.52 \cdot 0.48 / 1000} \approx 0.016$ — so $\hat{p}$ typically lands within about $\pm 3$ points of the truth (two standard errors, by the 95% rule). That "±3% margin of error" in news polls is exactly this computation.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

# Statistopia's vehicles: 1 = bicycle, 0 = car
vehicles = np.array([1, 0, 0, 1, 0, 1, 0, 0, 1, 0])
p = vehicles.mean()
print(f"population (1 = bike): {vehicles}")
print(f"population proportion p = {vehicles.sum()}/{len(vehicles)} = {p}")

### One sample, one estimate

In [ ]:
rng = np.random.default_rng(5)

sample = rng.choice(vehicles, size=6, replace=True)
p_hat = sample.mean()
print(f"sample of 6: {sample}")
print(f"sample proportion p_hat = {sample.sum()}/6 = {p_hat:.3f}")
print(f"(the video's sample had 2 bike owners: p_hat = {2/6:.3f})")

### $\hat{p}$ is an unbiased, shrinking-spread estimator — because it is a mean

In [ ]:
rng = np.random.default_rng(0)

print(f"{'n':>5} {'mean of p_hat':>14} {'std of p_hat':>13} {'sqrt(p(1-p)/n)':>15}")
for n in [6, 20, 100, 1000]:
    p_hats = vehicles[rng.integers(0, 10, size=(100_000, n))].mean(axis=1)
    print(f"{n:>5} {p_hats.mean():>14.4f} {p_hats.std():>13.4f} "
          f"{np.sqrt(p * (1 - p) / n):>15.4f}")
print(f"\nCentered on p = {p} at every n, with the Bernoulli-variance spread.")

### The count of successes is binomial

The number of bike owners in the sample follows Binomial$(n, p)$; dividing by $n$ gives $\hat{p}$'s exact (discrete) distribution.

In [ ]:
from math import comb
rng = np.random.default_rng(1)

n = 6
counts = vehicles[rng.integers(0, 10, size=(200_000, n))].sum(axis=1)

print(f"{'k bikes':>8} {'p_hat':>7} {'simulated':>10} {'Binomial(6, 0.4)':>17}")
for k in range(n + 1):
    theory = comb(n, k) * p**k * (1 - p)**(n - k)
    print(f"{k:>8} {k/n:>7.3f} {np.mean(counts == k):>10.4f} {theory:>17.4f}")

## 5. Visualizations

### The population picture

The video's ten islanders and their vehicles, with the bike owners highlighted.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
for i, v in enumerate(vehicles):
    color = "tab:green" if v == 1 else "tab:gray"
    label = "bike" if v == 1 else "car"
    ax.scatter(i, 0, s=900, color=color, alpha=0.8, edgecolor="black", zorder=3)
    ax.annotate(label, xy=(i, 0), ha="center", va="center", fontsize=8,
                color="white", fontweight="bold")
ax.set_title(f"Statistopia's vehicles: {vehicles.sum()} of {len(vehicles)} own a bike"
             f"  ->  p = {p}")
ax.set_xticks(range(10))
ax.set_xticklabels([f"person {i+1}" for i in range(10)], fontsize=8)
ax.set_yticks([])
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.show()

### The sampling distribution of $\hat{p}$: the ladder gets finer and tighter

In [ ]:
rng = np.random.default_rng(3)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
for ax, n in zip(axes, [6, 30, 300]):
    p_hats = vehicles[rng.integers(0, 10, size=(60_000, n))].mean(axis=1)
    ax.hist(p_hats, bins=min(n + 1, 50), density=True, color="tab:green",
            alpha=0.7, edgecolor="white")
    ax.axvline(p, color="tab:red", linestyle="--", linewidth=2, label=f"p = {p}")
    ax.set_title(f"n = {n}  (std {p_hats.std():.3f})")
    ax.set_xlabel("$\\hat{p}$")
    ax.set_xlim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("density")
plt.suptitle("Sample proportions cluster around p, tighter as n grows")
plt.tight_layout()
plt.show()

### Which proportions are hardest to estimate?

The spread $\sqrt{p(1-p)/n}$ peaks at $p = 0.5$: 50/50 populations are the noisiest, near-unanimous ones ($p$ near 0 or 1) the easiest — the Bernoulli-variance curve from Lesson 3, back in its natural habitat.

In [ ]:
p_grid = np.linspace(0, 1, 200)

plt.figure(figsize=(8.5, 4.5))
for n in [10, 100, 1000]:
    plt.plot(p_grid, np.sqrt(p_grid * (1 - p_grid) / n), linewidth=2,
             label=f"n = {n}")
plt.axvline(0.5, color="gray", linestyle=":", linewidth=1.5)
plt.title("Standard error of $\\hat{p}$: worst at p = 0.5, shrinking with n")
plt.xlabel("true proportion p")
plt.ylabel("std of $\\hat{p}$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** In a warehouse of $N = 500$ parts, 35 are defective. A quality check samples 40 parts and finds 2 defective. Compute $p$ and $\hat{p}$, and state which is the estimate of which.

**Exercise 2.** With $n = 6$, list every value $\hat{p}$ can take. Can any sample of 6 report exactly the true 40%? What is the smallest $n$ (other than multiples of 10) for which 0.4 is exactly representable?

### Intermediate

**Exercise 3.** Verify by enumeration (all $10^6$ equally likely with-replacement samples are too many — use the binomial instead) that $E[\hat{p}] = p$ for $n = 6$: compute $\sum_k \frac{k}{6}\,P(K = k)$ with $K \sim \text{Binomial}(6, 0.4)$.

**Exercise 4.** The pollster's rule of thumb: the margin of error of a poll is about $\frac{1}{\sqrt{n}}$ (two standard errors at the worst case $p = 0.5$). Check this for $n = 100, 400, 1000, 2500$, and find the $n$ needed for a ±2% margin.

### Advanced

**Exercise 5.** Biased sampling bites proportions too: suppose bike owners are *twice* as likely to be near the sampling spot as car owners. Simulate this over-sampling scheme and compute the average $\hat{p}$. How wrong is it, and does more data fix it?

**Exercise 6.** Two candidates at $p = 0.52$ vs. $0.48$: simulate polls of $n = 100$, $1000$, and $10{,}000$ and estimate the probability that the poll *calls the wrong leader* ($\hat{p} < 0.5$). How many people must you poll to be 95% sure of calling a 4-point race correctly?

## 7. Solutions

In [ ]:
# Exercise 1: parts and defects
print(f"Exercise 1: p = 35/500 = {35/500} (population proportion -- the truth)")
print(f"p_hat = 2/40 = {2/40} (sample proportion -- the estimate of p)")

In [ ]:
# Exercise 2: the ladder of p_hat values
print(f"Exercise 2: with n = 6, p_hat is one of {[f'{k}/6' for k in range(7)]}")
print("0.4 = 2/5 is not a multiple of 1/6, so no sample of 6 can say exactly 40%.")
print("Smallest non-multiple-of-10 n: n = 5 (2/5 = 0.4 exactly).")

In [ ]:
# Exercise 3: E[p_hat] via the binomial
from math import comb
expected = sum(k / 6 * comb(6, k) * 0.4**k * 0.6**(6 - k) for k in range(7))
print(f"Exercise 3: E[p_hat] = {expected:.4f} = p = 0.4 -- unbiased, exactly.")

In [ ]:
# Exercise 4: the 1/sqrt(n) margin of error
print("Exercise 4:")
for n in [100, 400, 1000, 2500]:
    print(f"  n = {n:>5}: 1/sqrt(n) = {1/np.sqrt(n):.3f} "
          f"(2 SE at p=0.5: {2 * np.sqrt(0.25 / n):.3f})")
n_needed = int(np.ceil(1 / 0.02**2))
print(f"+-2% needs 1/sqrt(n) = 0.02 -> n = {n_needed}")

In [ ]:
# Exercise 5: over-sampling bike owners
rng = np.random.default_rng(0)

# bike owners twice as likely to be sampled: weights 2 for bikes, 1 for cars
weights = np.where(vehicles == 1, 2.0, 1.0)
probs = weights / weights.sum()

for n in [6, 1000, 100_000]:
    sample = rng.choice(vehicles, size=n, replace=True, p=probs)
    print(f"n = {n:>6}: average p_hat = {sample.mean():.3f}")
print(f"\nTheory: 8/14 = {8/14:.3f}. The bias (0.571 vs 0.4) never shrinks --")
print("more data from a biased scheme just estimates the WRONG number more")
print("precisely. Only fixing the sampling fixes the estimate.")

In [ ]:
# Exercise 6: calling a 4-point race
rng = np.random.default_rng(1)

print("Exercise 6: P(poll shows the trailing candidate ahead), p = 0.52:")
for n in [100, 1000, 10_000]:
    p_hats = rng.binomial(n, 0.52, size=100_000) / n
    print(f"  n = {n:>6}: P(p_hat < 0.5) = {np.mean(p_hats < 0.5):.3f}")

for n in [500, 1000, 1700, 2000]:
    p_hats = rng.binomial(n, 0.52, size=100_000) / n
    if np.mean(p_hats > 0.5) >= 0.95:
        print(f"95% confidence of calling it right at n ~ {n}")
        break

## 8. Conclusion

- The **population proportion** $p = \frac{\#\{\text{items with the characteristic}\}}{N}$ (Statistopia's bikes: $\tfrac{4}{10} = 40\%$); the **sample proportion** $\hat{p}$ is the same ratio in a random sample ($\tfrac26 = 33.3\%$) and estimates $p$.
- A proportion is the **mean of a 0/1 variable**, so the sample-mean machinery transfers whole: $\hat{p}$ is unbiased, requires i.i.d. sampling (biased schemes stay wrong forever), and has standard error $\sqrt{p(1-p)/n}$ — worst at $p = 0.5$, shrinking like $1/\sqrt n$.
- The success *count* is Binomial$(n, p)$, so $\hat{p}$'s exact distribution is a rescaled binomial — Lesson 2 and Lesson 5 joining hands.
- Practical corollary: the pollster's $\pm 1/\sqrt{n}$ margin of error, and the price of precision ($\pm 2\%$ costs $n = 2500$).

Next: the estimator that *doesn't* transfer cleanly — the **sample variance** and its famous $n - 1$.